# EDA del catálogo audiovisual StreamView Analytics

## Historia que guía el análisis

StreamView necesita decidir qué contenidos destacar, qué géneros y países estudiar para futuras adquisiciones y cómo presentar el catálogo a sus distintas áreas. Para responder, seguiremos una secuencia sencilla:

1. Primero confirmaremos qué información contiene el catálogo y cuál es su alcance.
2. Después revisaremos si los datos están completos y son confiables para comparar.
3. A continuación observaremos cómo se distribuyen películas y series por géneros, países e idiomas.
4. Luego identificaremos los contenidos con mayor popularidad relativa y revisaremos la relación entre presupuesto e ingresos.
5. Finalmente convertiremos los hallazgos en una propuesta concreta para los dashboards.

Este notebook analiza exclusivamente `data/processed/catalogo_streamview.csv`, generado por `01_limpieza_union.ipynb`. No vuelve a limpiar ni modificar el catálogo maestro.

**Usuarios:** el Directorio necesita una visión resumida para decidir prioridades; Contenidos y Adquisición necesitan conocer la oferta y sus espacios; Marketing necesita localizar títulos y mercados con señales de interés.

**Regla de interpretación:** `popularity` es un índice relativo, no representa reproducciones, visualizaciones ni cantidad de usuarios.

## 1. ¿Con qué catálogo estamos trabajando?

Esta primera sección carga el archivo integrado y confirma su estructura, periodo y cantidad de contenidos. Sirve para que cualquier lector conozca el punto de partida antes de interpretar gráficos o tomar decisiones.

La unidad de análisis es un contenido audiovisual por fila. Las películas y las series se mantienen identificables mediante `type`.

In [21]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

working_dir = Path.cwd().resolve()
project_root = next((candidate for candidate in [working_dir, *working_dir.parents] if (candidate / 'data' / 'processed').is_dir()), None)
if project_root is None:
    raise FileNotFoundError('No se encontró la raíz del proyecto con data/processed/.')
processed_dir = project_root / 'data' / 'processed'
images_dir = project_root / 'images'
images_dir.mkdir(exist_ok=True)
catalog_path = processed_dir / 'catalogo_streamview.csv'
if not catalog_path.exists():
    raise FileNotFoundError('Falta catalogo_streamview.csv. Ejecuta primero 01_limpieza_union.ipynb.')

catalogo = pd.read_csv(catalog_path, low_memory=False)
expected_columns = ['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added', 'release_year', 'rating', 'duration', 'genres', 'language', 'popularity', 'vote_count', 'vote_average', 'budget', 'revenue']
missing_columns = set(expected_columns).difference(catalogo.columns)
if missing_columns:
    raise ValueError(f'Faltan columnas esperadas: {sorted(missing_columns)}')

catalogo['date_added'] = pd.to_datetime(catalogo['date_added'], errors='coerce')
for column in ['release_year', 'popularity', 'vote_count', 'vote_average', 'budget', 'revenue']:
    catalogo[column] = pd.to_numeric(catalogo[column], errors='coerce')
catalogo['added_year'] = catalogo['date_added'].dt.year
print(f'Archivo: {catalog_path}')
print(f'Filas: {len(catalogo):,} | Columnas originales: {len(expected_columns)}')
print(f'Periodo de incorporación: {int(catalogo.added_year.min())}-{int(catalogo.added_year.max())}')

Archivo: C:\Users\shein\OneDrive\Documentos\GitHub\visualizacion-de-datos-StreamView-Analytics\data\processed\catalogo_streamview.csv
Filas: 32,000 | Columnas originales: 17
Periodo de incorporación: 2010-2025


**Conclusión de la sección 1**  
El análisis parte de un catálogo integrado de **32.000 contenidos**, formado por películas y series, incorporados entre **2010 y 2025**. Esto nos permite comparar ambas categorías dentro de una misma fuente, manteniendo una fila por contenido. Antes de extraer conclusiones, debemos verificar qué tan completos y consistentes son sus campos.

## 2. ¿Qué tan confiables son los datos?

Antes de comparar países, géneros o popularidad, debemos saber si existen nulos, duplicados, valores fuera de rango o poca información financiera. Esta revisión evita presentar conclusiones engañosas.

Aquí solo diagnosticamos: las decisiones de limpieza ya pertenecen al notebook anterior. La cobertura financiera se calcula aparte porque `budget` y `revenue` solo aplican a películas.

In [22]:
quality_summary = pd.DataFrame({
    'tipo': catalogo[expected_columns].dtypes.astype(str),
    'nulos': catalogo[expected_columns].isna().sum(),
    'porcentaje_nulos': (catalogo[expected_columns].isna().mean() * 100).round(2),
    'valores_unicos': catalogo[expected_columns].nunique(dropna=True),
}).sort_values('porcentaje_nulos', ascending=False)

range_checks = {
    'release_year_fuera_1900_2025': ~catalogo['release_year'].between(1900, 2025),
    'rating_fuera_0_10': ~catalogo['rating'].between(0, 10),
    'popularity_negativa': catalogo['popularity'] < 0,
    'vote_count_negativo': catalogo['vote_count'] < 0,
    'budget_negativo': catalogo['budget'] < 0,
    'revenue_negativo': catalogo['revenue'] < 0,
}
range_audit = pd.DataFrame({'regla': list(range_checks), 'registros_invalidos': [int(mask.fillna(False).sum()) for mask in range_checks.values()]})
financial_coverage = pd.DataFrame({
    'metrica': ['peliculas_totales', 'con_budget', 'con_revenue', 'con_budget_y_revenue', 'cobertura'],
    'valor': [len(catalogo[catalogo.type == 'Movie']), int(catalogo.loc[catalogo.type == 'Movie', 'budget'].notna().sum()), int(catalogo.loc[catalogo.type == 'Movie', 'revenue'].notna().sum()), int(catalogo.loc[(catalogo.type == 'Movie') & catalogo.budget.notna() & catalogo.revenue.notna()].shape[0]), np.nan],
})
financial_coverage.loc[4, 'valor'] = financial_coverage.loc[3, 'valor'] / financial_coverage.loc[0, 'valor']

print('=== CALIDAD ===')
display(pd.Series({'filas': len(catalogo), 'duplicados_completos': int(catalogo.duplicated(expected_columns).sum()), 'show_id_repetidos': int(catalogo.show_id.duplicated().sum()), 'nulos_totales': int(catalogo[expected_columns].isna().sum().sum())}, name='valor').to_frame())
display(quality_summary)
print('=== RANGOS ===')
display(range_audit)
print('=== COBERTURA FINANCIERA ===')
display(financial_coverage)

=== CALIDAD ===


,valor
filas,32000
duplicados_completos,0
show_id_repetidos,406
nulos_totales,85310


,tipo,nulos,porcentaje_nulos,valores_unicos
budget,float64,27153,84.85,957
revenue,float64,26355,82.36,5326
duration,str,16000,50.00,1
director,str,11097,34.68,13011
country,str,2263,7.07,1745
cast,str,1361,4.25,30230
genres,str,1081,3.38,3895
show_id,int64,0,0.00,31594
title,str,0,0.00,30639
rating,float64,0,0.00,2728


=== RANGOS ===


,regla,registros_invalidos
0,release_year_fuera_1900_2025,0
1,rating_fuera_0_10,0
2,popularity_negativa,0
3,vote_count_negativo,0
4,budget_negativo,0
5,revenue_negativo,0


=== COBERTURA FINANCIERA ===


,metrica,valor
0,peliculas_totales,16000.00000
1,con_budget,4847.00000
2,con_revenue,5645.00000
3,con_budget_y_revenue,3540.00000
4,cobertura,0.22125


 **Conclusión de la sección 2**  
 El catálogo no está completamente informado: `budget` y `revenue` faltan en muchos registros porque son variables financieras de películas, y también existen campos faltantes en director, país y otros atributos. No se encontraron valores fuera de rango. Por eso, las comparaciones financieras deberán usar solo películas con datos válidos y las demás conclusiones deberán mostrar su cobertura.

## 3. ¿Cómo está compuesto el catálogo?

Ahora pasamos de la calidad a la composición. Separaremos géneros, países e idiomas en tablas derivadas para poder contarlos, pero sin alterar el catálogo maestro ni convertir un contenido en varias filas definitivas.

Esta sección sirve principalmente a Contenidos y Adquisición: permite reconocer concentraciones de oferta y posibles espacios de diferenciación.

In [23]:
def explode_dimension(dataframe, column):
    result = dataframe[['show_id', 'title', 'type', column, 'popularity', 'vote_average', 'vote_count']].copy()
    result[column] = result[column].fillna('Sin información').astype(str).str.split(r'\s*,\s*', regex=True)
    result = result.explode(column)
    result[column] = result[column].str.strip()
    return result[result[column].ne('')]

genres = explode_dimension(catalogo, 'genres')
countries = explode_dimension(catalogo, 'country')
languages = explode_dimension(catalogo, 'language')

def summarize_dimension(dataframe, dimension):
    valid = dataframe[dataframe[dimension].ne('Sin información')]
    return valid.groupby([dimension, 'type']).agg(contenidos=('show_id', 'nunique'), popularidad_promedio=('popularity', 'mean'), calificacion_promedio=('vote_average', 'mean')).reset_index()

genre_by_type = summarize_dimension(genres, 'genres')
country_by_type = summarize_dimension(countries, 'country')
language_by_type = languages[languages.language.ne('Sin información')].groupby(['language', 'type'])['show_id'].nunique().reset_index(name='contenidos')

kpis = pd.Series({
    'Total contenidos': len(catalogo),
    'Películas': int((catalogo.type == 'Movie').sum()),
    'Series': int((catalogo.type == 'TV Show').sum()),
    'Países': countries[countries.country.ne('Sin información')].country.nunique(),
    'Idiomas': languages[languages.language.ne('Sin información')].language.nunique(),
    'Géneros': genres[genres.genres.ne('Sin información')].genres.nunique(),
    'Popularidad promedio': catalogo.popularity.mean(),
    'Calificación promedio': catalogo.vote_average.mean(),
})
display(kpis.to_frame('valor').round(2))

,valor
Total contenidos,32000.00
Películas,16000.00
Series,16000.00
Países,147.00
Idiomas,83.00
Géneros,28.00
Popularidad promedio,42.62
Calificación promedio,5.69


 **Conclusión de la sección 3**  
 El catálogo ofrece diversidad: registra **147 países, 83 idiomas y 28 géneros**. Sin embargo, un mismo contenido puede pertenecer a varios géneros o países, por lo que estas cifras representan asociaciones y no deben sumarse como si fueran contenidos independientes. Esta estructura permite estudiar dónde existe concentración y dónde podría haber espacio para nuevas adquisiciones.

## 4. ¿Qué historias nos cuentan los contenidos?

Con la estructura clara, podemos responder las preguntas del caso. Compararemos películas y series por separado, porque sus escalas de popularidad y sus patrones de producción pueden ser distintos.

También revisaremos las finanzas solo en películas con información completa. De esta manera, una ausencia de datos no se confunde con un ingreso o presupuesto igual a cero.

El objetivo no es declarar ganadores definitivos, sino encontrar señales que orienten la adquisición, el marketing y el diseño del dashboard.

In [24]:
top_popularity_by_type = catalogo.dropna(subset=['popularity']).sort_values('popularity', ascending=False).groupby('type').head(10)
temporal_by_type = catalogo.dropna(subset=['added_year']).groupby(['added_year', 'type']).size().unstack(fill_value=0).sort_index()
financial = catalogo[(catalogo.type == 'Movie') & catalogo.budget.notna() & catalogo.revenue.notna() & (catalogo.budget > 0)].copy()
financial['profit'] = financial.revenue - financial.budget
financial['roi_approx'] = financial.profit / financial.budget
financial_kpis = pd.Series({
    'Películas financieras válidas': len(financial),
    'Presupuesto promedio': financial.budget.mean(),
    'Ingresos totales': financial.revenue.sum(),
    'ROI aproximado mediano': financial.roi_approx.median(),
    'Correlación presupuesto-ingresos': financial[['budget', 'revenue']].corr().iloc[0, 1],
})

country_summary = country_by_type.copy()
country_summary['participacion_catalogo'] = country_summary.contenidos / len(catalogo)
country_summary['popularidad_normalizada'] = country_summary.popularidad_promedio / country_summary.popularidad_promedio.max()
country_summary['oportunidad_proxy'] = country_summary.popularidad_normalizada * (1 - country_summary.participacion_catalogo)
market_priority = country_summary[country_summary.contenidos >= 30].sort_values('oportunidad_proxy', ascending=False)

print('=== TOP 10 POPULARIDAD POR TIPO ===')
display(top_popularity_by_type[['title', 'type', 'popularity', 'vote_average', 'vote_count', 'release_year']].round(2))
print('=== TOP 10 GÉNEROS POR TIPO ===')
display(genre_by_type.sort_values('contenidos', ascending=False).groupby('type').head(10).round(2))
print('=== TOP 10 PAÍSES POR TIPO ===')
display(country_by_type.sort_values('contenidos', ascending=False).groupby('type').head(10).round(2))
print('=== FINANZAS ===')
display(financial_kpis.to_frame('valor').round(3))
print('=== MERCADOS PROXY ===')
display(market_priority.head(10).round(3))

=== TOP 10 POPULARIDAD POR TIPO ===


,title,type,popularity,vote_average,vote_count,release_year
21000,The Late Show with Stephen Colbert,TV Show,6421.92,6.49,288,2015
20000,The Tonight Show Starring Jimmy Fallon,TV Show,4925.25,5.80,321,2014
15000,The Gorge,Movie,3876.01,7.78,1572,2025
18000,Good Mythical Morning,TV Show,3414.47,6.80,67,2012
15001,Flight Risk,Movie,3320.62,6.00,350,2025
24000,Chronicles of the Sun,TV Show,2987.48,6.80,117,2018
30000,Volta por Cima,TV Show,2885.72,5.78,16,2024
30001,She's the One,TV Show,2862.87,8.10,9,2024
14000,Mufasa: The Lion King,Movie,2643.63,7.47,1497,2024
30004,Crazy About You,TV Show,2442.24,5.20,16,2024


=== TOP 10 GÉNEROS POR TIPO ===


,genres,type,contenidos,popularidad_promedio,calificacion_promedio
12,Drama,TV Show,7859,64.13,5.72
11,Drama,Movie,6910,16.72,6.16
6,Comedy,TV Show,4572,66.33,6.06
5,Comedy,Movie,4533,19.42,6.07
30,Thriller,Movie,3769,24.49,5.84
0,Action,Movie,3239,32.21,6.07
24,Romance,Movie,2571,19.17,5.96
4,Animation,TV Show,2490,56.28,6.69
17,Horror,Movie,2425,21.45,5.44
1,Action & Adventure,TV Show,1988,67.40,6.76


=== TOP 10 PAÍSES POR TIPO ===


,country,type,contenidos,popularidad_promedio,calificacion_promedio
233,United States of America,Movie,7762,25.40,6.05
234,United States of America,TV Show,3193,84.01,6.68
44,China,TV Show,1900,43.79,5.31
107,Japan,TV Show,1879,49.43,6.68
230,United Kingdom,Movie,1743,20.89,6.14
72,France,Movie,1701,16.76,6.29
202,South Korea,TV Show,1299,65.29,6.14
37,Canada,Movie,1063,19.71,6.01
106,Japan,Movie,990,21.23,6.48
201,South Korea,Movie,875,17.30,5.63


=== FINANZAS ===


,valor
Películas financieras válidas,3.540000e+03
Presupuesto promedio,3.558214e+07
Ingresos totales,3.673714e+11
ROI aproximado mediano,7.000000e-01
Correlación presupuesto-ingresos,7.480000e-01


=== MERCADOS PROXY ===


,country,type,contenidos,popularidad_promedio,calificacion_promedio,participacion_catalogo,popularidad_normalizada,oportunidad_proxy
177,Portugal,TV Show,86,183.425,3.495,0.003,0.575,0.574
200,South Africa,TV Show,56,183.113,6.646,0.002,0.574,0.573
29,Brazil,TV Show,261,128.806,6.064,0.008,0.404,0.401
81,Greece,TV Show,59,111.342,6.112,0.002,0.349,0.349
46,Colombia,TV Show,90,101.306,6.228,0.003,0.318,0.317
141,Mexico,TV Show,315,95.499,6.008,0.010,0.300,0.297
93,India,TV Show,384,95.119,4.383,0.012,0.298,0.295
42,Chile,TV Show,117,93.472,3.394,0.004,0.293,0.292
152,Netherlands,TV Show,133,89.852,4.913,0.004,0.282,0.281
173,Philippines,TV Show,401,85.760,2.176,0.013,0.269,0.266


 **Conclusión de la sección 4**  
 Las señales más fuertes aparecen al separar películas y series: los géneros y países no tienen el mismo peso en ambas categorías, y el ranking de popularidad debe leerse junto con `vote_average` y `vote_count`. En las **3.540 películas con información financiera completa**, la correlación entre presupuesto e ingresos es positiva, pero el ROI aproximado debe interpretarse con cautela porque no representa la rentabilidad de StreamView. Estos resultados ya permiten pasar de preguntas generales a comparaciones concretas para cada audiencia.

## 5. ¿Qué gráficos ayudan a tomar decisiones?

Los gráficos de esta sección son prototipos para Looker Studio, no el dashboard final. Cada uno responde a una necesidad concreta: comparar popularidad, detectar concentración temática y geográfica, observar incorporaciones por año y estudiar la relación financiera.

Se usan paneles separados para películas y series. Así una categoría no oculta a la otra y el usuario puede leer cada comparación con mayor facilidad.

In [25]:
# Visualizaciones interactivas con Plotly, preparadas para explorar y reutilizar en dashboards.
import plotly.express as px

COLORS = {'Movie': '#e45756', 'TV Show': '#4c78a8'}
LABELS = {'Movie': 'Películas', 'TV Show': 'Series'}

# 1. Popularidad separada por tipo, ordenada de mayor a menor índice.
top_popularity_plot = top_popularity_by_type.copy()
top_popularity_plot['tipo_label'] = top_popularity_plot['type'].map(LABELS)
fig_popularity = px.bar(
    top_popularity_plot,
    x='popularity',
    y='title',
    color='tipo_label',
    facet_col='tipo_label',
    orientation='h',
    hover_data=['vote_average', 'vote_count', 'release_year'],
    color_discrete_map={'Películas': COLORS['Movie'], 'Series': COLORS['TV Show']},
    title='Top 10 contenidos por popularidad relativa y tipo',
    labels={'popularity': 'Índice de popularidad', 'title': '', 'tipo_label': 'Tipo'},
)
fig_popularity.update_layout(height=650, showlegend=False)
fig_popularity.update_yaxes(categoryorder='total descending')
fig_popularity.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split('=')[-1]))
fig_popularity.write_html(images_dir / 'netflix_eda_01_popularidad_tipo.html', include_plotlyjs='cdn')
fig_popularity.show()

# 2. Géneros por tipo, ordenados de mayor a menor cantidad de contenidos.
genre_plot = genre_by_type.copy()
genre_plot['tipo_label'] = genre_plot['type'].map(LABELS)
genre_plot = genre_plot.sort_values(['type', 'contenidos'], ascending=[True, False]).groupby('type').head(10)
fig_genres = px.bar(
    genre_plot,
    x='contenidos',
    y='genres',
    color='tipo_label',
    facet_col='tipo_label',
    orientation='h',
    hover_data=['popularidad_promedio', 'calificacion_promedio'],
    color_discrete_map={'Películas': COLORS['Movie'], 'Series': COLORS['TV Show']},
    title='Géneros dominantes por tipo de contenido',
    labels={'contenidos': 'Contenidos asociados', 'genres': '', 'tipo_label': 'Tipo'},
)
fig_genres.update_layout(height=650, showlegend=False)
fig_genres.update_yaxes(categoryorder='total descending')
fig_genres.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split('=')[-1]))
fig_genres.write_html(images_dir / 'netflix_eda_02_generos_tipo.html', include_plotlyjs='cdn')
fig_genres.show()

# 3. Países por tipo, ordenados de mayor a menor cantidad de contenidos.
country_plot = country_by_type.copy()
country_plot['tipo_label'] = country_plot['type'].map(LABELS)
country_plot = country_plot.sort_values(['type', 'contenidos'], ascending=[True, False]).groupby('type').head(10)
fig_countries = px.bar(
    country_plot,
    x='contenidos',
    y='country',
    color='tipo_label',
    facet_col='tipo_label',
    orientation='h',
    hover_data=['popularidad_promedio', 'calificacion_promedio'],
    color_discrete_map={'Películas': COLORS['Movie'], 'Series': COLORS['TV Show']},
    title='Países con mayor presencia por tipo de contenido',
    labels={'contenidos': 'Contenidos asociados', 'country': '', 'tipo_label': 'Tipo'},
)
fig_countries.update_layout(height=650, showlegend=False)
fig_countries.update_yaxes(categoryorder='total descending')
fig_countries.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split('=')[-1]))
fig_countries.write_html(images_dir / 'netflix_eda_03_paises_tipo.html', include_plotlyjs='cdn')
fig_countries.show()

# 4. Evolución temporal: paneles independientes por tipo.
temporal_plot = temporal_by_type.reset_index().melt(
    id_vars='added_year', var_name='type', value_name='contenidos'
)
temporal_plot['tipo_label'] = temporal_plot['type'].map(LABELS)
fig_temporal = px.line(
    temporal_plot,
    x='added_year',
    y='contenidos',
    color='tipo_label',
    facet_row='tipo_label',
    markers=True,
    color_discrete_map={'Películas': COLORS['Movie'], 'Series': COLORS['TV Show']},
    title='Evolución anual de incorporaciones por tipo',
    labels={'added_year': 'Año de incorporación', 'contenidos': 'Contenidos', 'tipo_label': 'Tipo'},
)
fig_temporal.update_layout(height=650, showlegend=False)
fig_temporal.for_each_annotation(lambda annotation: annotation.update(text=annotation.text.split('=')[-1]))
fig_temporal.write_html(images_dir / 'netflix_eda_04_temporal_tipo.html', include_plotlyjs='cdn')
fig_temporal.show()

# 5. Finanzas: hover con ROI y escalas logarítmicas para comparar rangos amplios.
financial_plot = financial[financial['revenue'] > 0].copy()
fig_finances = px.scatter(
    financial_plot,
    x='budget',
    y='revenue',
    color='roi_approx',
    hover_data=['title', 'release_year', 'profit'],
    color_continuous_scale='RdBu_r',
    log_x=True,
    log_y=True,
    title='Relación entre presupuesto e ingresos',
    labels={'budget': 'Budget, escala logarítmica', 'revenue': 'Revenue, escala logarítmica', 'roi_approx': 'ROI aproximado'},
)
fig_finances.update_layout(height=650)
fig_finances.write_html(images_dir / 'netflix_eda_05_finanzas.html', include_plotlyjs='cdn')
fig_finances.show()

print('Visualizaciones Plotly exportadas como HTML en:', images_dir)

Visualizaciones Plotly exportadas como HTML en: C:\Users\shein\OneDrive\Documentos\GitHub\visualizacion-de-datos-StreamView-Analytics\images


 **Conclusión de la sección 5**  
 Los prototipos confirman que no necesitamos mostrar muchos gráficos: necesitamos mostrar los que permitan comparar. El ranking separado ayuda a Marketing a encontrar títulos; los paneles de géneros y países ayudan a Contenidos a detectar concentración; la línea temporal permite revisar incorporaciones por tipo; y la dispersión financiera ayuda al Directorio a explorar riesgo. La línea temporal puede verse plana porque los datos tienen una distribución uniforme por año; eso debe explicarse y no presentarse como crecimiento real.

## 6. ¿Cómo convertimos los hallazgos en un dashboard?

El EDA termina transformando preguntas en decisiones visuales. La tabla siguiente sirve como puente hacia Looker Studio: indica qué usuario necesita la información, qué pregunta debe responder el gráfico, qué variables requiere y qué acción puede apoyar.

La propuesta evita llenar el dashboard de gráficos decorativos. Cada visual debe existir porque ayuda a comparar, detectar una oportunidad o justificar una decisión.

In [26]:
dashboard_plan = pd.DataFrame([
    {'audiencia': 'Directorio', 'pregunta': '¿Cuál es el tamaño y valor del catálogo?', 'visual': 'KPI cards + tabla financiera', 'variables': 'type, vote_average, budget, revenue', 'filtros': 'tipo, año', 'decision': 'priorizar inversión'},
    {'audiencia': 'Contenidos/Adquisición', 'pregunta': '¿Qué géneros y países están saturados o subrepresentados?', 'visual': 'small multiples de barras', 'variables': 'genres, country, type, contenidos', 'filtros': 'tipo, idioma, año', 'decision': 'buscar oportunidades de adquisición'},
    {'audiencia': 'Marketing', 'pregunta': '¿Qué títulos tienen mayor interés relativo?', 'visual': 'ranking horizontal', 'variables': 'title, popularity, vote_count, vote_average', 'filtros': 'tipo, género, país, año', 'decision': 'seleccionar campañas'},
    {'audiencia': 'Directorio/Finanzas', 'pregunta': '¿La inversión se relaciona con ingresos?', 'visual': 'dispersión logarítmica', 'variables': 'budget, revenue, ROI', 'filtros': 'año, rango financiero', 'decision': 'evaluar riesgo'},
])

display(dashboard_plan)
print('Plan disponible en memoria para diseñar el dashboard.')
print('El EDA no exporta tablas CSV auxiliares; se conserva únicamente catalogo_streamview.csv.')

,audiencia,pregunta,visual,variables,filtros,decision
0,Directorio,¿Cuál es el tamaño y valor del catálogo?,KPI cards + tabla financiera,"type, vote_average, budget, revenue","tipo, año",priorizar inversión
1,Contenidos/Adquisición,¿Qué géneros y países están saturados o subrep...,small multiples de barras,"genres, country, type, contenidos","tipo, idioma, año",buscar oportunidades de adquisición
2,Marketing,¿Qué títulos tienen mayor interés relativo?,ranking horizontal,"title, popularity, vote_count, vote_average","tipo, género, país, año",seleccionar campañas
3,Directorio/Finanzas,¿La inversión se relaciona con ingresos?,dispersión logarítmica,"budget, revenue, ROI","año, rango financiero",evaluar riesgo


Plan disponible en memoria para diseñar el dashboard.
El EDA no exporta tablas CSV auxiliares; se conserva únicamente catalogo_streamview.csv.


 **Conclusión de la sección 6**  
 El dashboard debe organizarse alrededor de decisiones, no de una colección de gráficos. El Directorio necesita KPIs y riesgo financiero; Contenidos necesita comparar géneros y países por tipo; Marketing necesita rankings con contexto. Los filtros de tipo, año, país y género permitirán explorar sin perder la historia principal. El siguiente paso es implementar esta propuesta en Looker Studio y validar si cada usuario encuentra la respuesta que necesita.